# 前処理結果 簡単確認

このノートブックでは、前処理結果を簡単に確認します。

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

# 実験設定
EXPERIMENT_NAME = "preprocess_v2_ws64"  # 変更してください
DATA_DIR = Path("../output/experiments")
EXPERIMENT_DIR = DATA_DIR / EXPERIMENT_NAME

print(f"実験: {EXPERIMENT_NAME}")
print(f"ディレクトリ: {EXPERIMENT_DIR}")
print(f"存在: {EXPERIMENT_DIR.exists()}")

実験: preprocess_v2_ws64
ディレクトリ: ../output/experiments/preprocess_v2_ws64
存在: True


In [2]:
# ファイル確認
files = {
    'windows': 'train_windows.pkl',
    'demographics': 'train_demographics.pkl', 
    'tabular': 'train_tabular.pkl',
    'tof_voxel': 'train_tof_voxel.pkl',
    'tof_windows': 'train_tof_windows.pkl',
    'labels': 'train_labels.pkl',
    'info': 'train_info.pkl'
}

print("📁 ファイル確認:")
for key, filename in files.items():
    filepath = EXPERIMENT_DIR / 'preprocessed' / filename
    if filepath.exists():
        file_size = filepath.stat().st_size / (1024 * 1024)  # MB
        print(f"  ✅ {filename}: {file_size:.1f} MB")
    else:
        print(f"  ❌ {filename}: 見つかりません")

📁 ファイル確認:
  ✅ train_windows.pkl: 44.6 MB
  ✅ train_demographics.pkl: 0.3 MB
  ✅ train_tabular.pkl: 30.3 MB
  ✅ train_tof_voxel.pkl: 701.8 MB
  ✅ train_tof_windows.pkl: 792.7 MB
  ✅ train_labels.pkl: 0.1 MB
  ✅ train_info.pkl: 0.3 MB


In [3]:
# データ読み込みと基本統計
print("📊 データ統計:")

for key, filename in files.items():
    filepath = EXPERIMENT_DIR / 'preprocessed' / filename
    if filepath.exists():
        try:
            with open(filepath, 'rb') as f:
                data = pickle.load(f)
            
            if hasattr(data, 'shape'):
                print(f"\n  {key}:")
                print(f"    形状: {data.shape}")
                if np.issubdtype(data.dtype, np.number):
                    print(f"    範囲: [{data.min():.3f}, {data.max():.3f}]")
                    print(f"    平均: {data.mean():.3f}")
                    print(f"    標準偏差: {data.std():.3f}")
                    
                    # 欠損値チェック
                    if np.isnan(data).any():
                        missing_count = np.isnan(data).sum()
                        print(f"    ⚠️ 欠損値: {missing_count}個")
                    else:
                        print(f"    ✅ 欠損値: なし")
            
            elif isinstance(data, (list, np.ndarray)):
                print(f"\n  {key}:")
                print(f"    形状: {np.array(data).shape}")
                if isinstance(data, list) and len(data) > 0:
                    print(f"    要素数: {len(data)}")
                    print(f"    最初の要素: {type(data[0])}")
            
            else:
                print(f"\n  {key}: {type(data)}")
                
        except Exception as e:
            print(f"\n  {key}: 読み込みエラー - {e}")

📊 データ統計:

  windows:
    形状: (10147, 64, 18)
    範囲: [-17.230, 18.970]
    平均: -0.000
    標準偏差: 1.000
    ✅ 欠損値: なし

  demographics:
    形状: (10147, 7)
    範囲: [-3.024, 6.302]
    平均: -0.000
    標準偏差: 1.000
    ✅ 欠損値: なし

  tabular:
    形状: (10147, 392)
    範囲: [-16.475, 100.727]
    平均: 0.000
    標準偏差: 0.860
    ✅ 欠損値: なし

  tof_voxel:
    形状: (574945, 5, 8, 8)
    範囲: [-0.864, 2.336]
    平均: -0.000
    標準偏差: 1.000
    ✅ 欠損値: なし

  tof_windows:
    形状: (10147, 64, 5, 8, 8)
    範囲: [-0.864, 2.336]
    平均: -0.088
    標準偏差: 0.988
    ✅ 欠損値: なし

  labels:
    形状: (10147,)
    範囲: [0.000, 17.000]
    平均: 8.117
    標準偏差: 5.062
    ✅ 欠損値: なし

  info:
    形状: (10147,)
    要素数: 10147
    最初の要素: <class 'dict'>


In [4]:
# ラベル分布確認
labels_file = EXPERIMENT_DIR / 'preprocessed' / 'train_labels.pkl'
if labels_file.exists():
    try:
        with open(labels_file, 'rb') as f:
            labels = pickle.load(f)
        
        print(f"\n🏷️ ラベル分布:")
        unique, counts = np.unique(labels, return_counts=True)
        print(f"  総サンプル数: {len(labels)}")
        print(f"  ラベル数: {len(unique)}")
        
        for label, count in zip(unique, counts):
            percentage = count / len(labels) * 100
            print(f"  ラベル {label}: {count} サンプル ({percentage:.1f}%)")
        
        # クラス不均衡チェック
        min_count = counts.min()
        max_count = counts.max()
        imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
        print(f"  クラス不均衡比: {imbalance_ratio:.2f}")
        
        if imbalance_ratio > 10:
            print(f"  ⚠️ クラス不均衡が大きいです")
        else:
            print(f"  ✅ クラス分布は比較的均等です")
            
    except Exception as e:
        print(f"\n🏷️ ラベル読み込みエラー: {e}")
else:
    print("\n🏷️ ラベルファイルが見つかりません")


🏷️ ラベル分布:
  総サンプル数: 10147
  ラベル数: 18
  ラベル 0: 702 サンプル (6.9%)
  ラベル 1: 700 サンプル (6.9%)
  ラベル 2: 219 サンプル (2.2%)
  ラベル 3: 804 サンプル (7.9%)
  ラベル 4: 697 サンプル (6.9%)
  ラベル 5: 307 サンプル (3.0%)
  ラベル 6: 705 サンプル (6.9%)
  ラベル 7: 704 サンプル (6.9%)
  ラベル 8: 243 サンプル (2.4%)
  ラベル 9: 705 サンプル (6.9%)
  ラベル 10: 1105 サンプル (10.9%)
  ラベル 11: 175 サンプル (1.7%)
  ラベル 12: 519 サンプル (5.1%)
  ラベル 13: 229 サンプル (2.3%)
  ラベル 14: 1019 サンプル (10.0%)
  ラベル 15: 600 サンプル (5.9%)
  ラベル 16: 541 サンプル (5.3%)
  ラベル 17: 173 サンプル (1.7%)
  クラス不均衡比: 6.39
  ✅ クラス分布は比較的均等です


In [5]:
# サンプル数整合性チェック
print("\n🔍 サンプル数整合性チェック:")

sample_counts = {}
for key, filename in files.items():
    filepath = EXPERIMENT_DIR / 'preprocessed' / filename
    if filepath.exists():
        try:
            with open(filepath, 'rb') as f:
                data = pickle.load(f)
            
            if hasattr(data, 'shape'):
                sample_counts[key] = data.shape[0]
            elif isinstance(data, (list, np.ndarray)):
                sample_counts[key] = len(data)
        except:
            pass

if sample_counts:
    base_count = list(sample_counts.values())[0]
    for key, count in sample_counts.items():
        status = "✅" if count == base_count else "❌"
        print(f"  {status} {key}: {count}")
else:
    print("  データを読み込めませんでした")


🔍 サンプル数整合性チェック:
  ✅ windows: 10147
  ✅ demographics: 10147
  ✅ tabular: 10147
  ❌ tof_voxel: 574945
  ✅ tof_windows: 10147
  ✅ labels: 10147
  ✅ info: 10147


In [6]:
print("\n✅ 簡単確認完了！")
print("\n📝 次のステップ:")
print("  1. 詳細な分析が必要な場合は 'preprocessing_analysis.ipynb' を実行")
print("  2. 問題がある場合は前処理スクリプトを再実行")
print("  3. 問題がない場合は学習に進む")


✅ 簡単確認完了！

📝 次のステップ:
  1. 詳細な分析が必要な場合は 'preprocessing_analysis.ipynb' を実行
  2. 問題がある場合は前処理スクリプトを再実行
  3. 問題がない場合は学習に進む
